In [0]:
from utils.helper import BatchControlHelper
from pyspark.sql import DataFrame
from pyspark.sql import functions as f
import json
from pyspark.sql.window import Window
from delta.tables import DeltaTable

In [0]:
dbutils.widgets.text("environment", "dev")

dbutils.widgets.text("processing_date", "2024-12-31", "As-of date used when generating the initial load (YYYY-MM-DD)")
dbutils.widgets.dropdown("force_reload", "false", ["false", "true"], "Reprocess tables even if already SUCCESS")

processing_date = dbutils.widgets.get("processing_date")
force_reload = dbutils.widgets.get("force_reload") == "true"
env = dbutils.widgets.get("environment")

batch_id = f"ods-reports-{processing_date.replace('-', '')}"


## 1. Current transaction operations view


In [0]:
transaction = spark.sql(f"SELECT * FROM {env}_banking.ods.transaction")
transaction = (transaction.withColumn('row_effective_at', f.col('source_modified_at'))
    .withColumn('row_effective_at', f.col('source_modified_at'))
    .withColumn('event_arrival_lag_days', f.when(
        f.col('arrival_timestamp').isNotNull() & f.col('transaction_timestamp').isNotNull(),
        f.greatest(f.lit(0), f.datediff(f.to_date('arrival_timestamp'), f.col('transaction_timestamp')))))
    .withColumn('is_late_arrival',
        f.coalesce(f.col('is_late_arrival').cast('boolean'), f.lit(False)) |
        (f.col('arrival_timestamp').isNotNull() & f.col('transaction_timestamp').isNotNull() &
         (f.to_date('arrival_timestamp') > f.col('transaction_timestamp'))))
)

account_context = spark.sql(f"SELECT * FROM {env}_banking.ods.account").alias('a')
branch_context = spark.sql(f"SELECT * FROM {env}_banking.ods.branch").select('branch_id', 'branch_name', 'region').alias('b')
customer_today = (spark.sql(f"SELECT * FROM {env}_banking.ods.customer_history")
    .filter((f.col('row_effective_date') <= processing_date) &
            (f.col('row_expiration_date').isNull() | (processing_date < f.col('row_expiration_date'))))
    .select('customer_id', 'customer_segment', 'kyc_status'))

transaction_operational = (transaction.alias('t')
    .join(account_context, f.col('t.account_id') == f.col('a.account_id'), 'left')
    .join(branch_context, f.col('a.branch_id') == f.col('b.branch_id'), 'left')
    .join(customer_today.alias('c'), f.col('a.primary_customer_id') == f.col('c.customer_id'), 'left')
    .select(f.col('t.transaction_hk'), f.col('t.transaction_id'),
        f.col('t.account_id'), f.col('a.primary_customer_id'), f.col('c.customer_segment'), f.col('c.kyc_status'),
        f.col('a.account_type'), f.col('a.account_status'), f.col('a.branch_id'), f.col('b.branch_name'), f.col('b.region'),
        f.col('t.transaction_timestamp'), f.col('t.transaction_date'),
        f.col('t.transaction_type'), f.col('t.transaction_amount').alias('transaction_amount'),
        f.col('t.currency_code'), f.col('t.transaction_status'), f.col('t.merchant_name'), f.col('t.merchant_category'),
        f.col('t.arrival_timestamp'), f.to_date('t.arrival_timestamp').alias('arrival_date'), f.col('t.event_arrival_lag_days'),
        f.col('t.row_effective_at')))

In [0]:
transaction_operational.write.format('delta').mode('overwrite').option('overwriteSchema', 'true').saveAsTable(f'{env}_banking.ods.operational_transactions_latest')

## 2. Account operating position


In [0]:
balance_window = Window.partitionBy('account_id').orderBy(f.col('balance_date').desc(), f.col('source_modified_at').desc_nulls_last())
latest_balance = (spark.sql(f"SELECT * FROM {env}_banking.ods.daily_account_balance")
    .filter(f.to_date('balance_date') <= processing_date)  # remove future data balances
    .withColumn('_balance_rank', f.row_number().over(balance_window)).filter(f.col('_balance_rank') == 1)
    .select('account_id', f.to_date('balance_date').alias('balance_date'),
        f.col('available_balance_amount').alias('available_balance_amount'),
        f.col('ledger_balance_amount').alias('ledger_balance_amount'),
        f.col('overdraft_limit_amount').alias('overdraft_limit_amount')))


account_txn = spark.sql(f"SELECT * FROM {env}_banking.ods.operational_transactions_latest")

activity_30d = (account_txn
    .filter(f.col('transaction_date').between(f.date_sub(f.lit(processing_date), 29), f.lit(processing_date)))
    .groupBy('account_id')
    .agg(f.count('*').alias('transactions_last_30d'),
        f.sum(f.when(f.col('transaction_status') == 'POSTED', 1).otherwise(0)).alias('posted_transactions_last_30d'),
        f.sum(f.when(f.col('transaction_status') == 'REVERSED', 1).otherwise(0)).alias('reversed_transactions_last_30d'),
        f.sum(f.when((f.col('transaction_status') == 'POSTED') & (f.col('transaction_amount') > 0), f.col('transaction_amount'))
              .otherwise(f.lit(0))).alias('credits_last_30d'),
        f.sum(f.when((f.col('transaction_status') == 'REVERSED') & (f.col('transaction_amount') > 0), -f.col('transaction_amount'))
              .otherwise(f.lit(0))).alias('reversed_last_30d'),
        f.sum(f.when((f.col('transaction_status') == 'POSTED') & (f.col('transaction_amount') < 0), -f.col('transaction_amount'))
              .otherwise(f.lit(0))).alias('debits_last_30d')))


last_activity_window = Window.partitionBy('account_id').orderBy(f.col('transaction_timestamp').desc_nulls_last(), f.col('transaction_id').desc())
last_account_activity = (account_txn.withColumn('_activity_rank', f.row_number().over(last_activity_window)).filter(f.col('_activity_rank') == 1)
    .select('account_id', f.col('transaction_timestamp').alias('last_transaction_at'),
        f.col('transaction_status').alias('last_transaction_status')))

account_operational = (spark.sql(f"SELECT * FROM {env}_banking.ods.account").alias('a')
    .join(spark.sql(f"SELECT * FROM {env}_banking.ods.branch").select('branch_id', 'branch_name', 'region').alias('b'),
        f.col('a.branch_id') == f.col('b.branch_id'), 'left')
    .join(customer_today.alias('c'), f.col('a.primary_customer_id') == f.col('c.customer_id'), 'left')
    .join(latest_balance.alias('bal'), f.col('a.account_id') == f.col('bal.account_id'), 'left')
    .join(activity_30d.alias('act'), f.col('a.account_id') == f.col('act.account_id'), 'left')
    .join(last_account_activity.alias('last'), f.col('a.account_id') == f.col('last.account_id'), 'left')
    .select(f.col('a.account_id'), f.col('a.primary_customer_id'), f.col('c.customer_segment'), f.col('c.kyc_status'),
        f.col('a.branch_id'), f.col('b.branch_name'), f.col('b.region'), f.col('a.account_type'), f.col('a.account_status'),
        f.col('a.open_date'), f.col('a.close_date'), f.col('a.currency_code'), f.col('bal.balance_date'),
        f.col('bal.available_balance_amount'), f.col('bal.ledger_balance_amount'), f.col('bal.overdraft_limit_amount'),
        f.coalesce(f.col('act.transactions_last_30d'), f.lit(0)).alias('transactions_last_30d'),
        f.coalesce(f.col('act.posted_transactions_last_30d'), f.lit(0)).alias('posted_transactions_last_30d'),
        f.coalesce(f.col('act.credits_last_30d'), f.lit(0)).alias('credits_last_30d'),
        f.coalesce(f.col('act.debits_last_30d'), f.lit(0)).alias('debits_last_30d'),
        f.coalesce(f.col('act.reversed_transactions_last_30d'), f.lit(0)).alias('reversed_transactions_last_30d'),
        f.coalesce(f.col('act.reversed_last_30d'), f.lit(0)).alias('reversed_last_30d'),
        f.col('last.last_transaction_at'), f.col('last.last_transaction_status')))


In [0]:
account_operational.write.format('delta').mode('overwrite').option('overwriteSchema', 'true').saveAsTable(f'{env}_banking.ods.operational_accounts')

## 3. Event-date Branch reporting metrics


In [0]:
operational_txns = spark.sql(f"SELECT * FROM {env}_banking.ods.operational_transactions_latest")
daily_metrics = (operational_txns
    .groupBy('transaction_date', 'branch_id')
    .agg(f.countDistinct('account_id').alias('num_accounts_transacted'),f.count('*').alias('transaction_count'), 
        f.sum(f.when(f.col('transaction_type') == 'CARD_PURCHASE', 1).otherwise(0)).alias('cnt_cards_purchased'),
        f.sum(f.when(f.col('transaction_type') == 'DIRECT_DEPOSIT', 1).otherwise(0)).alias('cnt_direct_deposits'),
        f.sum(f.when(f.col('transaction_type') == 'ACH_DEBIT', 1).otherwise(0)).alias('cnt_ach_debits'),
        f.sum(f.when(f.col('transaction_type') == 'ATM_WITHDRAWAL', 1).otherwise(0)).alias('cnt_atm_withdrawls'),
        f.sum(f.when(f.col('transaction_type') == 'TRANSFER', 1).otherwise(0)).alias('cnt_transfers'),
        f.sum(f.when(f.col('transaction_status') == 'POSTED', 1).otherwise(0)).alias('cnt_posted_transactions'),
        f.sum(f.when(f.col('transaction_status') == 'REVERSED', 1).otherwise(0)).alias('cnt_reversed_transactions'),
        f.sum(f.when(f.col('transaction_amount') > 0, f.col('transaction_amount')).otherwise(f.lit(0))).alias('credit_amount'),
        f.sum(f.when(f.col('transaction_amount') < 0, -f.col('transaction_amount')).otherwise(f.lit(0))).alias('debit_amount'),
        f.sum('transaction_amount').alias('net_amount'),
        f.sum(f.when((f.col('transaction_status') == 'POSTED') & (f.col('transaction_amount') > 0), f.col('transaction_amount')).otherwise(f.lit(0))).alias('posted_credit_amount'),
        f.sum(f.when((f.col('transaction_status') == 'POSTED') & (f.col('transaction_amount') < 0), -f.col('transaction_amount')).otherwise(f.lit(0))).alias('posted_debit_amount'),
        f.sum(f.when(f.col('transaction_status') == 'POSTED', f.col('transaction_amount')).otherwise(f.lit(0))).alias('posted_net_amount'))
        # f.sum(f.when(f.col('is_late_arrival') & (f.col('transaction_status') == 'POSTED'), f.col('transaction_amount')).otherwise(f.lit(0))).alias('late_arrival_posted_net_amount'))
    .withColumnRenamed('transaction_date', 'business_date'))
# replace_delta_table(daily_metrics, 'ods_operational_daily_metrics')

In [0]:
daily_metrics.write.format('delta').mode('overwrite').option('overwriteSchema', 'true').saveAsTable(f'{env}_banking.ods.operational_branch_daily_metrics')

In [0]:
display(spark.table(f"{env}_banking.ods.operational_branch_daily_metrics").orderBy(f.col('business_date').desc(), f.col('branch_id')).limit(100))


In [0]:
display(spark.table(f"{env}_banking.ods.operational_accounts").orderBy(f.col('last_transaction_at').asc()).limit(100))

In [0]:
display(spark.table(f"{env}_banking.ods.operational_transactions_latest").orderBy(f.col('transaction_timestamp').desc()).limit(100))
print('ODS reporting models passed Silver-grain, current-key and temporal-range checks.')